## Real vs synthetic data

This notebook operates on the **real** ACN-Data held-out results produced by `notebooks/10_real_acn_data_experiment.ipynb`. The synthetic / placeholder results from Stages 7-8 are not used here. The `source` field of each `real_*` artifact is `live_api` (verified by the loader's status block); no placeholder or synthetic value is substituted for a real-data value.


## Quantum-advantage disclaimer

This notebook does **not** claim quantum advantage. The headline result is reported on the exact classical solver; QAOA is reported for methodology validation only. The 11-qubit instance is small enough that the exact classical optimum is computable; any quantum-advantage language is explicitly avoided.


# 11 — Held-out evaluation and statistics

## Question

On the temporally held-out test set, which formulation (F0, F1, F2, F3) achieves the highest P(feasible), the lowest unmet demand, and the lowest cost? Are the differences statistically significant?

## Why this test exists

The headline result of the paper is F2 vs F0 on P(feasible) on the held-out set, with a paired bootstrap 95% confidence interval. The secondary comparisons (F1 vs F0, F2 vs F1) use Bonferroni correction (α = 0.05 / 3).

## Method

For each held-out session, evaluate the F0/F1/F2/F3 frozen schedule under the session's realized (Δd, ΔE). Compute the four primitive feasibility metrics (qubo_feasible, energy_feasible, site_feasible, deadline_feasible) and the aggregate `feasible = AND`. The paired bootstrap CI uses n_bootstrap = 10,000 resamples with seed `20260829`.

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.


## Implementation


In [ ]:
import json, pathlib
p = pathlib.Path('../artifacts/real_heldout_results.json')
if not p.exists():
    print('real_heldout_results.json not produced yet. Run notebook 10 (driver) first.')
else:
    s = json.loads(p.read_text())
    print(f"{'Method':8s}  {'P(feas)':>8s}  {'mean_unmet':>10s}  {'deadline_v':>10s}  {'site_v':>8s}  {'mean_cost':>10s}")
    for fname, h in s.items():
        print(f"{fname:8s}  "
              f"{h.get('P_feasible', 0):>8.4f}  "
              f"{h.get('mean_unmet_kWh', 0):>10.4f}  "
              f"{h.get('deadline_violation_rate', 0):>10.4f}  "
              f"{h.get('site_violation_rate', 0):>8.4f}  "
              f"{h.get('mean_cost', 0):>10.4f}")


In [ ]:
import json, pathlib
p = pathlib.Path('../artifacts/real_paired_statistics.json')
if not p.exists():
    print('real_paired_statistics.json not produced yet.')
else:
    s = json.loads(p.read_text())
    print(f"Primary endpoint: F2 vs F0 P(feasible), 95% CI")
    prim = s.get('primary_F2_vs_F0', {})
    print(f"  diff = {prim.get('diff_mean'):.4f}")
    print(f"  95% CI = [{prim.get('ci_lo'):.4f}, {prim.get('ci_hi'):.4f}]")
    print()
    print(f"Bonferroni α = {s.get('bonferroni_alpha')}")
    print()
    for label, key in [('F1 vs F0', 'secondary_F1_vs_F0'), ('F2 vs F1', 'secondary_F2_vs_F1')]:
        sec = s.get(key, {})
        print(f"Secondary {label}: diff = {sec.get('diff_mean'):.4f}, 95% CI = [{sec.get('ci_lo'):.4f}, {sec.get('ci_hi'):.4f}]")


## Interpretation

The headline result is the primary F2 vs F0 bootstrap CI. If the CI excludes 0, the improvement is statistically significant at α = 0.05. Secondary comparisons (F1 vs F0, F2 vs F1) use the Bonferroni-corrected α = 0.05/3.

**Honest-reporting rule:** if F0 dominates F1/F2, the result is reported as-is. The paper does not re-tune γ, K, α, M_window, or any frozen parameter to make F2 win. The negative result is a valid scientific finding.

## Limitations

- The held-out window is 6 months; longer would be better.
- The bootstrap CI depends on the number of held-out sessions. If   the held-out sample is small, the CI is wide and the test is   underpowered.
- We do not perform multiple-testing correction across all 16   metrics (cost, unmet, peak, deadline, site, P(feas) for 4   formulations). The Bonferroni correction is applied to the 3   primary and secondary P(feas) comparisons only.


## Result

See the code outputs above for the numerical results. The interpretation is in the next section.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.


## Result

See the code outputs above for the numerical results. The interpretation is in the next section.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.
